In [1]:
import os
from itertools import combinations
from gurobipy import GRB
import sys
sys.path.insert(0, '../../scripts')
import io_utils as io
import post_processing as ppr

We will now compare all pairs of bins using our post-processing method (with read depth in the objective function). 

The output for this test is available in USRA_PLASMIDS_2023/data/POST-PROCESS-ALL-PAIRS-2023-08-01/post_processing_all_bins.tsv.

In [2]:
columns = [
    'SAMPLE', 'BIN1', 'BIN1_CONTIGS', 'BIN1_GC_BIN', 'BIN1_GC', 'BIN1_GD', 'BIN1_RD',
    'BIN2', 'BIN2_CONTIGS', 'BIN2_GC_BIN', 'BIN2_GC', 'BIN2_GD', 'BIN2_RD', 'MERGE_GC_BIN',
    'MERGE_GC', 'MERGE_GD', 'MERGE_RD', 'MERGE_PATH', 'MILP_INFEASIBLE', 'BINS_OVERLAP'
]

# merge function closure
def generate_merge_function(input_dir, output_dir, gc_file, model_solution_dir, out_tsv_file):
    
    with open(out_tsv_file, 'w') as file:
        file.write('\t'.join(columns))
    # returns string containing contig ID
    # and given properties of that contig
    # separated by colons
    def ctg_to_str(ctg, *args):

        return ':'.join([ctg, *map(lambda x: f'{x.get(ctg)}', args)])

    # returns string representing set of contigs
    def ctgs_to_str(ctgs, *args):

        return ','.join([ctg_to_str(ctg, *args) for ctg in ctgs])

    # returns relevant data from post-processing results
    def post_process_merge_data(model, b1, b2, pbf_input, pbf_output):

        gc_bin, gc_term, gd_term, rd_term, path_contigs = 0, 0, 0, 0, []
        was_solved = model.getAttr('Status') == GRB.OPTIMAL 
        b1_gc_bin = pbf_output.get_gc_bin(b1)
        b1_gc = model.getVarByName('ctg_GC[s,{}]'.format(b1_gc_bin)).Obj
        b2_gc_bin = pbf_output.get_gc_bin(b2)
        b2_gc = model.getVarByName('ctg_GC[t,{}]'.format(b2_gc_bin)).Obj
        b1_ctgs = pbf_output.get_pls_ctgs(b1, False)
        b2_ctgs = pbf_output.get_pls_ctgs(b2, False)
        overlap = set(b1_ctgs) & set(b2_ctgs) != set()
        ctg_lens = pbf_input.get_ctg_lengths()
        b1_ctgs_str = ctgs_to_str(b1_ctgs, ctg_lens)
        b2_ctgs_str = ctgs_to_str(b2_ctgs, ctg_lens)
        flows = pbf_output.get_flows()

        if was_solved:
            rd_term = model.getVarByName('d').X
            for v in model.getVars():
                if v.X > 0.0001:
                    if v.VarName[:2] == 'GC':
                        gc_bin = int(v.VarName[3])
                    elif v.VarName[:3] == 'ctg':
                        gc_term += v.Obj
                    elif v.VarName[0] == 'z':
                        gd_term += v.Obj
                        if v.VarName[2:-1] not in ['s', 't']:
                            path_contigs.append(v.VarName[2:-1])

        path_contigs = ctgs_to_str(path_contigs, ctg_lens)
        col_vals = [
            b1, b1_ctgs_str, b1_gc_bin, b1_gc, model.getVarByName('z[s]').Obj, flows[b1],
            b2, b2_ctgs_str, b2_gc_bin, b2_gc, model.getVarByName('z[t]').Obj, flows[b2],
            gc_bin, gc_term, gd_term, rd_term, path_contigs, int(not was_solved), overlap
        ]
        return col_vals

    # Merges given bins and outputs list containing results
    def merge_bins(sample, b1, b2, pbf_input, pbf_output, threshold):

        b1_ctgs = pbf_output.get_pls_ctgs(b1, False)
        b2_ctgs = pbf_output.get_pls_ctgs(b2, False)
        sol_path = os.path.join(model_solution_dir, '_'.join([sample, b1, b2]) + '.sol')    
        model = ppr.post_processing_model(sample, b1, b2, input_dir, output_dir, gc_file, threshold)
        model.setParam('OutputFlag', False)
        if os.path.isfile(sol_path):
            model.update()
            model.read(sol_path)
        model.optimize()
        if model.getAttr('Status') == GRB.OPTIMAL and not os.path.isfile(sol_path):
                model.write(sol_path)
        line_data = post_process_merge_data(model, b1, b2, pbf_input, pbf_output)

        return [sample, *line_data]


    def merge_all_pairs(sample, threshold):

        pbf_input = io.PBF_input(os.path.join(input_dir, sample, 'assembly.gfa'), gc_file, \
                                     os.path.join(input_dir, sample, 'filtered_genes_to_contigs.csv'), \
                                     0.58, 2650)
        pbf_output = io.PBF_output('plasbin_flow', os.path.join(output_dir, sample, 'plasbin_flow_bins.out'), gc_file)
        pred_bins = pbf_output.get_pls()
        for b1, b2 in combinations(pred_bins.get_pls_ids(), 2):
            print('Merging', b1, b2, 'in', sample)
            line = merge_bins(sample, b1, b2, pbf_input, pbf_output, threshold)
            with open(out_tsv_file, 'a') as file:
                file.write('\n' + '\t'.join(['{}'.format(data) for data in line]))
    
    return merge_all_pairs

In [3]:
input_dir = '../../../plasbin_flow_input'
output_dir = '../../../organized_data/output'
gc_file = '../../../organized_data/gc_intervals.txt'
model_solution_dir = './post_processing_results'
out_tsv_file = 'post_processing_all_bins.tsv'
sample_merger = generate_merge_function(input_dir, output_dir, gc_file, model_solution_dir, out_tsv_file)
for sample in os.listdir(output_dir):
    sample_merger(sample, 0.05)

Merging pbf_P0 pbf_P1 in sample_102
Set parameter Username
Academic license - for non-commercial use only - expires 2024-06-27
Merging pbf_P0 pbf_P2 in sample_102
Merging pbf_P1 pbf_P2 in sample_102
Merging pbf_P0 pbf_P1 in sample_107
Merging pbf_P0 pbf_P2 in sample_107
Merging pbf_P0 pbf_P3 in sample_107
Merging pbf_P1 pbf_P2 in sample_107
Merging pbf_P1 pbf_P3 in sample_107
Merging pbf_P2 pbf_P3 in sample_107
Merging pbf_P0 pbf_P1 in sample_108
Merging pbf_P0 pbf_P2 in sample_108
Merging pbf_P1 pbf_P2 in sample_108
Merging pbf_P0 pbf_P1 in sample_109
Merging pbf_P0 pbf_P2 in sample_109
Merging pbf_P0 pbf_P3 in sample_109
Merging pbf_P0 pbf_P4 in sample_109
Merging pbf_P0 pbf_P5 in sample_109
Merging pbf_P1 pbf_P2 in sample_109
Merging pbf_P1 pbf_P3 in sample_109
Merging pbf_P1 pbf_P4 in sample_109
Merging pbf_P1 pbf_P5 in sample_109
Merging pbf_P2 pbf_P3 in sample_109
Merging pbf_P2 pbf_P4 in sample_109
Merging pbf_P2 pbf_P5 in sample_109
Merging pbf_P3 pbf_P4 in sample_109
Merging p

Merging pbf_P17 pbf_P23 in sample_15
Merging pbf_P17 pbf_P25 in sample_15
Merging pbf_P18 pbf_P22 in sample_15
Merging pbf_P18 pbf_P23 in sample_15
Merging pbf_P18 pbf_P25 in sample_15
Merging pbf_P22 pbf_P23 in sample_15
Merging pbf_P22 pbf_P25 in sample_15
Merging pbf_P23 pbf_P25 in sample_15
Merging pbf_P0 pbf_P1 in sample_16
Merging pbf_P0 pbf_P2 in sample_16
Merging pbf_P0 pbf_P3 in sample_16
Merging pbf_P0 pbf_P4 in sample_16
Merging pbf_P0 pbf_P5 in sample_16
Merging pbf_P0 pbf_P6 in sample_16
Merging pbf_P0 pbf_P8 in sample_16
Merging pbf_P1 pbf_P2 in sample_16
Merging pbf_P1 pbf_P3 in sample_16
Merging pbf_P1 pbf_P4 in sample_16
Merging pbf_P1 pbf_P5 in sample_16
Merging pbf_P1 pbf_P6 in sample_16
Merging pbf_P1 pbf_P8 in sample_16
Merging pbf_P2 pbf_P3 in sample_16
Merging pbf_P2 pbf_P4 in sample_16
Merging pbf_P2 pbf_P5 in sample_16
Merging pbf_P2 pbf_P6 in sample_16
Merging pbf_P2 pbf_P8 in sample_16
Merging pbf_P3 pbf_P4 in sample_16
Merging pbf_P3 pbf_P5 in sample_16
Merg

Merging pbf_P1 pbf_P8 in sample_25
Merging pbf_P1 pbf_P9 in sample_25
Merging pbf_P1 pbf_P10 in sample_25
Merging pbf_P1 pbf_P11 in sample_25
Merging pbf_P1 pbf_P12 in sample_25
Merging pbf_P1 pbf_P13 in sample_25
Merging pbf_P2 pbf_P3 in sample_25
Merging pbf_P2 pbf_P4 in sample_25
Merging pbf_P2 pbf_P5 in sample_25
Merging pbf_P2 pbf_P6 in sample_25
Merging pbf_P2 pbf_P7 in sample_25
Merging pbf_P2 pbf_P8 in sample_25
Merging pbf_P2 pbf_P9 in sample_25
Merging pbf_P2 pbf_P10 in sample_25
Merging pbf_P2 pbf_P11 in sample_25
Merging pbf_P2 pbf_P12 in sample_25
Merging pbf_P2 pbf_P13 in sample_25
Merging pbf_P3 pbf_P4 in sample_25
Merging pbf_P3 pbf_P5 in sample_25
Merging pbf_P3 pbf_P6 in sample_25
Merging pbf_P3 pbf_P7 in sample_25
Merging pbf_P3 pbf_P8 in sample_25
Merging pbf_P3 pbf_P9 in sample_25
Merging pbf_P3 pbf_P10 in sample_25
Merging pbf_P3 pbf_P11 in sample_25
Merging pbf_P3 pbf_P12 in sample_25
Merging pbf_P3 pbf_P13 in sample_25
Merging pbf_P4 pbf_P5 in sample_25
Merging 

Merging pbf_P1 pbf_P7 in sample_33
Merging pbf_P1 pbf_P9 in sample_33
Merging pbf_P1 pbf_P10 in sample_33
Merging pbf_P1 pbf_P12 in sample_33
Merging pbf_P1 pbf_P13 in sample_33
Merging pbf_P1 pbf_P14 in sample_33
Merging pbf_P1 pbf_P15 in sample_33
Merging pbf_P1 pbf_P16 in sample_33
Merging pbf_P1 pbf_P17 in sample_33
Merging pbf_P2 pbf_P3 in sample_33
Merging pbf_P2 pbf_P4 in sample_33
Merging pbf_P2 pbf_P6 in sample_33
Merging pbf_P2 pbf_P7 in sample_33
Merging pbf_P2 pbf_P9 in sample_33
Merging pbf_P2 pbf_P10 in sample_33
Merging pbf_P2 pbf_P12 in sample_33
Merging pbf_P2 pbf_P13 in sample_33
Merging pbf_P2 pbf_P14 in sample_33
Merging pbf_P2 pbf_P15 in sample_33
Merging pbf_P2 pbf_P16 in sample_33
Merging pbf_P2 pbf_P17 in sample_33
Merging pbf_P3 pbf_P4 in sample_33
Merging pbf_P3 pbf_P6 in sample_33
Merging pbf_P3 pbf_P7 in sample_33
Merging pbf_P3 pbf_P9 in sample_33
Merging pbf_P3 pbf_P10 in sample_33
Merging pbf_P3 pbf_P12 in sample_33
Merging pbf_P3 pbf_P13 in sample_33
Mer

Merging pbf_P5 pbf_P8 in sample_42
Merging pbf_P5 pbf_P9 in sample_42
Merging pbf_P7 pbf_P8 in sample_42
Merging pbf_P7 pbf_P9 in sample_42
Merging pbf_P8 pbf_P9 in sample_42
Merging pbf_P0 pbf_P1 in sample_44
Merging pbf_P0 pbf_P2 in sample_44
Merging pbf_P0 pbf_P3 in sample_44
Merging pbf_P0 pbf_P4 in sample_44
Merging pbf_P0 pbf_P5 in sample_44
Merging pbf_P0 pbf_P6 in sample_44
Merging pbf_P1 pbf_P2 in sample_44
Merging pbf_P1 pbf_P3 in sample_44
Merging pbf_P1 pbf_P4 in sample_44
Merging pbf_P1 pbf_P5 in sample_44
Merging pbf_P1 pbf_P6 in sample_44
Merging pbf_P2 pbf_P3 in sample_44
Merging pbf_P2 pbf_P4 in sample_44
Merging pbf_P2 pbf_P5 in sample_44
Merging pbf_P2 pbf_P6 in sample_44
Merging pbf_P3 pbf_P4 in sample_44
Merging pbf_P3 pbf_P5 in sample_44
Merging pbf_P3 pbf_P6 in sample_44
Merging pbf_P4 pbf_P5 in sample_44
Merging pbf_P4 pbf_P6 in sample_44
Merging pbf_P5 pbf_P6 in sample_44
Merging pbf_P0 pbf_P1 in sample_45
Merging pbf_P0 pbf_P2 in sample_45
Merging pbf_P0 pbf_P

Merging pbf_P4 pbf_P9 in sample_56
Merging pbf_P4 pbf_P10 in sample_56
Merging pbf_P4 pbf_P11 in sample_56
Merging pbf_P4 pbf_P12 in sample_56
Merging pbf_P4 pbf_P13 in sample_56
Merging pbf_P4 pbf_P14 in sample_56
Merging pbf_P5 pbf_P6 in sample_56
Merging pbf_P5 pbf_P7 in sample_56
Merging pbf_P5 pbf_P8 in sample_56
Merging pbf_P5 pbf_P9 in sample_56
Merging pbf_P5 pbf_P10 in sample_56
Merging pbf_P5 pbf_P11 in sample_56
Merging pbf_P5 pbf_P12 in sample_56
Merging pbf_P5 pbf_P13 in sample_56
Merging pbf_P5 pbf_P14 in sample_56
Merging pbf_P6 pbf_P7 in sample_56
Merging pbf_P6 pbf_P8 in sample_56
Merging pbf_P6 pbf_P9 in sample_56
Merging pbf_P6 pbf_P10 in sample_56
Merging pbf_P6 pbf_P11 in sample_56
Merging pbf_P6 pbf_P12 in sample_56
Merging pbf_P6 pbf_P13 in sample_56
Merging pbf_P6 pbf_P14 in sample_56
Merging pbf_P7 pbf_P8 in sample_56
Merging pbf_P7 pbf_P9 in sample_56
Merging pbf_P7 pbf_P10 in sample_56
Merging pbf_P7 pbf_P11 in sample_56
Merging pbf_P7 pbf_P12 in sample_56
Me

Merging pbf_P9 pbf_P10 in sample_87


We will now run the post-processing after dropping out contigs with residual read depth (i.e. after removing read depth used up by other plasmids bins) less than the minimum of the read depth assigned to either bin to be merged.

In [4]:
input_dir = '../../../plasbin_flow_input'
output_dir = '../../../organized_data/output'
gc_file = '../../../organized_data/gc_intervals.txt'
model_solution_dir = './post_processing_results_min_rd'
out_tsv_file = 'post_processing_all_bins_min_rd.tsv'
sample_merger = generate_merge_function(input_dir, output_dir, gc_file, model_solution_dir, out_tsv_file)

for sample in os.listdir(output_dir):
    sample_merger(sample, 'min')

Merging pbf_P0 pbf_P1 in sample_102
Merging pbf_P0 pbf_P2 in sample_102
Merging pbf_P1 pbf_P2 in sample_102
Merging pbf_P0 pbf_P1 in sample_107
Merging pbf_P0 pbf_P2 in sample_107
Merging pbf_P0 pbf_P3 in sample_107
Merging pbf_P1 pbf_P2 in sample_107
Merging pbf_P1 pbf_P3 in sample_107
Merging pbf_P2 pbf_P3 in sample_107
Merging pbf_P0 pbf_P1 in sample_108
Merging pbf_P0 pbf_P2 in sample_108
Merging pbf_P1 pbf_P2 in sample_108
Merging pbf_P0 pbf_P1 in sample_109
Merging pbf_P0 pbf_P2 in sample_109
Merging pbf_P0 pbf_P3 in sample_109
Merging pbf_P0 pbf_P4 in sample_109
Merging pbf_P0 pbf_P5 in sample_109
Merging pbf_P1 pbf_P2 in sample_109
Merging pbf_P1 pbf_P3 in sample_109
Merging pbf_P1 pbf_P4 in sample_109
Merging pbf_P1 pbf_P5 in sample_109
Merging pbf_P2 pbf_P3 in sample_109
Merging pbf_P2 pbf_P4 in sample_109
Merging pbf_P2 pbf_P5 in sample_109
Merging pbf_P3 pbf_P4 in sample_109
Merging pbf_P3 pbf_P5 in sample_109
Merging pbf_P4 pbf_P5 in sample_109
Merging pbf_P0 pbf_P1 in sam

Merging pbf_P18 pbf_P22 in sample_15
Merging pbf_P18 pbf_P23 in sample_15
Merging pbf_P18 pbf_P25 in sample_15
Merging pbf_P22 pbf_P23 in sample_15
Merging pbf_P22 pbf_P25 in sample_15
Merging pbf_P23 pbf_P25 in sample_15
Merging pbf_P0 pbf_P1 in sample_16
Merging pbf_P0 pbf_P2 in sample_16
Merging pbf_P0 pbf_P3 in sample_16
Merging pbf_P0 pbf_P4 in sample_16
Merging pbf_P0 pbf_P5 in sample_16
Merging pbf_P0 pbf_P6 in sample_16
Merging pbf_P0 pbf_P8 in sample_16
Merging pbf_P1 pbf_P2 in sample_16
Merging pbf_P1 pbf_P3 in sample_16
Merging pbf_P1 pbf_P4 in sample_16
Merging pbf_P1 pbf_P5 in sample_16
Merging pbf_P1 pbf_P6 in sample_16
Merging pbf_P1 pbf_P8 in sample_16
Merging pbf_P2 pbf_P3 in sample_16
Merging pbf_P2 pbf_P4 in sample_16
Merging pbf_P2 pbf_P5 in sample_16
Merging pbf_P2 pbf_P6 in sample_16
Merging pbf_P2 pbf_P8 in sample_16
Merging pbf_P3 pbf_P4 in sample_16
Merging pbf_P3 pbf_P5 in sample_16
Merging pbf_P3 pbf_P6 in sample_16
Merging pbf_P3 pbf_P8 in sample_16
Merging 

Merging pbf_P1 pbf_P10 in sample_25
Merging pbf_P1 pbf_P11 in sample_25
Merging pbf_P1 pbf_P12 in sample_25
Merging pbf_P1 pbf_P13 in sample_25
Merging pbf_P2 pbf_P3 in sample_25
Merging pbf_P2 pbf_P4 in sample_25
Merging pbf_P2 pbf_P5 in sample_25
Merging pbf_P2 pbf_P6 in sample_25
Merging pbf_P2 pbf_P7 in sample_25
Merging pbf_P2 pbf_P8 in sample_25
Merging pbf_P2 pbf_P9 in sample_25
Merging pbf_P2 pbf_P10 in sample_25
Merging pbf_P2 pbf_P11 in sample_25
Merging pbf_P2 pbf_P12 in sample_25
Merging pbf_P2 pbf_P13 in sample_25
Merging pbf_P3 pbf_P4 in sample_25
Merging pbf_P3 pbf_P5 in sample_25
Merging pbf_P3 pbf_P6 in sample_25
Merging pbf_P3 pbf_P7 in sample_25
Merging pbf_P3 pbf_P8 in sample_25
Merging pbf_P3 pbf_P9 in sample_25
Merging pbf_P3 pbf_P10 in sample_25
Merging pbf_P3 pbf_P11 in sample_25
Merging pbf_P3 pbf_P12 in sample_25
Merging pbf_P3 pbf_P13 in sample_25
Merging pbf_P4 pbf_P5 in sample_25
Merging pbf_P4 pbf_P6 in sample_25
Merging pbf_P4 pbf_P7 in sample_25
Merging 

Merging pbf_P1 pbf_P10 in sample_33
Merging pbf_P1 pbf_P12 in sample_33
Merging pbf_P1 pbf_P13 in sample_33
Merging pbf_P1 pbf_P14 in sample_33
Merging pbf_P1 pbf_P15 in sample_33
Merging pbf_P1 pbf_P16 in sample_33
Merging pbf_P1 pbf_P17 in sample_33
Merging pbf_P2 pbf_P3 in sample_33
Merging pbf_P2 pbf_P4 in sample_33
Merging pbf_P2 pbf_P6 in sample_33
Merging pbf_P2 pbf_P7 in sample_33
Merging pbf_P2 pbf_P9 in sample_33
Merging pbf_P2 pbf_P10 in sample_33
Merging pbf_P2 pbf_P12 in sample_33
Merging pbf_P2 pbf_P13 in sample_33
Merging pbf_P2 pbf_P14 in sample_33
Merging pbf_P2 pbf_P15 in sample_33
Merging pbf_P2 pbf_P16 in sample_33
Merging pbf_P2 pbf_P17 in sample_33
Merging pbf_P3 pbf_P4 in sample_33
Merging pbf_P3 pbf_P6 in sample_33
Merging pbf_P3 pbf_P7 in sample_33
Merging pbf_P3 pbf_P9 in sample_33
Merging pbf_P3 pbf_P10 in sample_33
Merging pbf_P3 pbf_P12 in sample_33
Merging pbf_P3 pbf_P13 in sample_33
Merging pbf_P3 pbf_P14 in sample_33
Merging pbf_P3 pbf_P15 in sample_33
M

Merging pbf_P7 pbf_P8 in sample_42
Merging pbf_P7 pbf_P9 in sample_42
Merging pbf_P8 pbf_P9 in sample_42
Merging pbf_P0 pbf_P1 in sample_44
Merging pbf_P0 pbf_P2 in sample_44
Merging pbf_P0 pbf_P3 in sample_44
Merging pbf_P0 pbf_P4 in sample_44
Merging pbf_P0 pbf_P5 in sample_44
Merging pbf_P0 pbf_P6 in sample_44
Merging pbf_P1 pbf_P2 in sample_44
Merging pbf_P1 pbf_P3 in sample_44
Merging pbf_P1 pbf_P4 in sample_44
Merging pbf_P1 pbf_P5 in sample_44
Merging pbf_P1 pbf_P6 in sample_44
Merging pbf_P2 pbf_P3 in sample_44
Merging pbf_P2 pbf_P4 in sample_44
Merging pbf_P2 pbf_P5 in sample_44
Merging pbf_P2 pbf_P6 in sample_44
Merging pbf_P3 pbf_P4 in sample_44
Merging pbf_P3 pbf_P5 in sample_44
Merging pbf_P3 pbf_P6 in sample_44
Merging pbf_P4 pbf_P5 in sample_44
Merging pbf_P4 pbf_P6 in sample_44
Merging pbf_P5 pbf_P6 in sample_44
Merging pbf_P0 pbf_P1 in sample_45
Merging pbf_P0 pbf_P2 in sample_45
Merging pbf_P0 pbf_P3 in sample_45
Merging pbf_P0 pbf_P4 in sample_45
Merging pbf_P0 pbf_P